# Week 3 — Data Preprocessing
**Internship:** IDX Exchange Data Science Program  
**Name:** Monika  
**Week:** 3  
**Dataset:** CRMLS Sold Properties (Jan 2022 – June 2026)

**Goal:** Handle missing values properly (impute + flag rather than drop), encode
categoricals, engineer features, normalize, and set up a flexible train/test split
mechanism ahead of the Week 4 baseline model.

In [30]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', 100)

data_folder = r'C:\Users\monik\OneDrive - University of Illinois - Urbana\Desktop\IDX Exchange_DS\data\california'

file_list = [
    'CRMLSSold20220101_20231231_filled.csv',
    'CRMLSSold202401_filled.csv', 'CRMLSSold202402_filled.csv', 'CRMLSSold202403_filled.csv',
    'CRMLSSold202404_filled.csv', 'CRMLSSold202405_filled.csv', 'CRMLSSold202406_filled.csv',
    'CRMLSSold202407_filled.csv', 'CRMLSSold202408.csv', 'CRMLSSold202409.csv',
    'CRMLSSold202410.csv', 'CRMLSSold202411.csv', 'CRMLSSold202412.csv',
    'CRMLSSold202501_filled.csv', 'CRMLSSold202502.csv', 'CRMLSSold202503.csv',
    'CRMLSSold202504.csv', 'CRMLSSold202505.csv', 'CRMLSSold202506.csv',
    'CRMLSSold202507.csv', 'CRMLSSold202508.csv', 'CRMLSSold202509.csv',
    'CRMLSSold202510.csv', 'CRMLSSold202511.csv', 'CRMLSSold202512.csv',
    'CRMLSSold202601.csv', 'CRMLSSold202602.csv', 'CRMLSSold202603.csv',
    'CRMLSSold202604.csv', 'CRMLSSold202605.csv',
    'CRMLSSold202606.csv',
]

dfs = [pd.read_csv(data_folder + '\\' + f, low_memory=False) for f in file_list]
df_raw = pd.concat(dfs, ignore_index=True)
print(f'Total rows loaded: {len(df_raw):,}')

Total rows loaded: 818,778


## 1. Re-apply Week 2 Filtering
Same duplicate drop + SingleFamilyResidence filter as Week 2, so this notebook is
self-contained and reproducible from raw files.

In [31]:
df_raw['CloseDate_parsed'] = pd.to_datetime(df_raw['CloseDate'], errors='coerce')
df_raw = df_raw.drop_duplicates(subset=['ListingKey'], keep='first')

df = df_raw[
    (df_raw['PropertyType'] == 'Residential') &
    (df_raw['PropertySubType'] == 'SingleFamilyResidence')
].copy()

print(f'Rows after filter: {len(df):,}')

Rows after filter: 411,729


## 2. Drop Rows With Invalid or Unrecoverable ClosePrice
ClosePrice is my target. If it's missing or non-positive (e.g. the $1 sales that
are likely quitclaim/family transfers, not real market transactions), the row
can't be used for training or evaluation at all — this isn't the same as
"missing data I could impute," so dropping here is justified.

In [32]:
before = len(df)
df = df[df['ClosePrice'].notna() & (df['ClosePrice'] > 0)].copy()
print(f'Dropped {before - len(df):,} rows with missing/non-positive ClosePrice')
print(f'Remaining: {len(df):,}')

Dropped 3 rows with missing/non-positive ClosePrice
Remaining: 411,726


## 3. Cap ClosePrice Outliers
Week 2's audit flagged ~3.14% of rows outside 3×IQR bounds, including a $989.5M
sale and several sub-$50k sales that are almost certainly not standard
arms-length transactions. Rather than just noting this like last time, I'm
capping ClosePrice at a floor of $50,000 (below which sales are very unlikely
to reflect real market value) and at the 3xIQR upper bound, rather than dropping
these rows outright — capping preserves more data than dropping while still
preventing extreme values from distorting the regression.

In [33]:
MIN_PRICE = 50_000

q1 = df['ClosePrice'].quantile(0.25)
q3 = df['ClosePrice'].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 3 * iqr

before = len(df)
df = df[df['ClosePrice'] >= MIN_PRICE].copy()
print(f'Dropped {before - len(df):,} rows below ${MIN_PRICE:,} floor')

n_capped = (df['ClosePrice'] > upper_bound).sum()
df['ClosePrice'] = df['ClosePrice'].clip(upper=upper_bound)
print(f'Capped {n_capped:,} rows at upper bound ${upper_bound:,.0f}')

Dropped 115 rows below $50,000 floor
Capped 12,918 rows at upper bound $3,840,000


## 4. Missing Value Decisions

| Column | Missing % (Wk2 audit) | Decision |
|---|---|---|
| LivingArea | 0.05% | Median impute + `LivingArea_missing` flag |
| BedroomsTotal | 0.00% | No action needed |
| BathroomsTotalInteger | 0.02% | Median impute + flag |
| LotSizeAcres | 1.72% | Median impute **by City**, fallback to global median + flag |
| YearBuilt | 0.08% | Median impute, then converted to PropertyAge below |
| Latitude / Longitude | 0.05% | Drop rows (can't reasonably impute geolocation) |
| PoolPrivateYN | small | Missing → False |
| ViewYN, WaterfrontYN, BasementYN | 80–99% | Missing → False (treated as "not applicable") |
| AssociationFee | moderate | Missing → 0 (no HOA fee reported) |
| DaysOnMarket | 0.00% but has negatives (min -265) | Clip at 0, add `DaysOnMarket_anomaly` flag |

I'm imputing + flagging instead of dropping wherever the missing % is low, so the
model can still learn "this value was imputed" as a signal if it matters, without
throwing away rows.

In [34]:
before = len(df)
df = df[df['Latitude'].notna() & df['Longitude'].notna()].copy()
print(f'Dropped {before - len(df):,} rows missing Latitude/Longitude')

Dropped 192 rows missing Latitude/Longitude


In [35]:
def impute_with_flag(frame, col):
    flag_col = f'{col}_missing'
    frame[flag_col] = frame[col].isna().astype(int)
    frame[col] = frame[col].fillna(frame[col].median())
    return frame

for col in ['LivingArea', 'BathroomsTotalInteger', 'YearBuilt']:
    df = impute_with_flag(df, col)

# LotSizeAcres: impute by City median first, fall back to global median
df['LotSizeAcres_missing'] = df['LotSizeAcres'].isna().astype(int)
city_median = df.groupby('City')['LotSizeAcres'].transform('median')
df['LotSizeAcres'] = df['LotSizeAcres'].fillna(city_median)
df['LotSizeAcres'] = df['LotSizeAcres'].fillna(df['LotSizeAcres'].median())

print(df[['LivingArea_missing', 'BathroomsTotalInteger_missing',
          'YearBuilt_missing', 'LotSizeAcres_missing']].sum())

LivingArea_missing                214
BathroomsTotalInteger_missing      75
YearBuilt_missing                 317
LotSizeAcres_missing             7089
dtype: int64


In [36]:
df['DaysOnMarket_anomaly'] = (df['DaysOnMarket'] < 0).astype(int)
print(f'Negative DaysOnMarket rows flagged: {df["DaysOnMarket_anomaly"].sum():,}')
df['DaysOnMarket'] = df['DaysOnMarket'].clip(lower=0)

Negative DaysOnMarket rows flagged: 87


In [37]:
bool_cols = ['PoolPrivateYN', 'ViewYN', 'WaterfrontYN', 'BasementYN']
for col in bool_cols:
    df[col] = df[col].fillna(False).astype(bool)

C:\Users\monik\AppData\Local\Temp\ipykernel_14888\3758494198.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False).astype(bool)
C:\Users\monik\AppData\Local\Temp\ipykernel_14888\3758494198.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False).astype(bool)
C:\Users\monik\AppData\Local\Temp\ipykernel_14888\3758494198.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To op

In [38]:
df['AssociationFee'] = df['AssociationFee'].fillna(0)

## 5. Feature Engineering: PropertyAge
Same as Week 2's plan — replacing YearBuilt with PropertyAge, which is a more
directly interpretable feature (and won't confuse the model the way a raw year
value might, since price relationships to "year built" aren't linear across a
250-year range).

In [39]:
CURRENT_YEAR = 2026
df['PropertyAge'] = CURRENT_YEAR - df['YearBuilt']
df.loc[df['PropertyAge'] < 0, 'PropertyAge'] = 0  # guard against bad future-dated YearBuilt

## 6. Encode Remaining Categoricals
The boolean fields are already 0/1-ready. City/PostalCode are too high-cardinality
for one-hot encoding directly — I'm leaving them out of the model features for now
but keeping City around since I used it for the LotSizeAcres imputation above and
it may come back later (e.g. group-based features or the Week 6 school-district join).

In [40]:
bool_int_cols = ['PoolPrivateYN', 'ViewYN', 'WaterfrontYN', 'BasementYN']
for col in bool_int_cols:
    df[col] = df[col].astype(int)

## 7. Final Feature Set
Deliberately **excluding ListPrice and OriginalListPrice** as model features.
This is meant to be a Zestimate-style valuation model that predicts price from
property characteristics — using ListPrice as an input would mean the model is
mostly just learning "trust the agent's number," which defeats the purpose and
would make the R² look artificially strong.

In [41]:
feature_cols = [
    'LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeAcres',
    'PropertyAge', 'DaysOnMarket', 'Latitude', 'Longitude',
    'PoolPrivateYN', 'ViewYN', 'WaterfrontYN', 'BasementYN', 'AssociationFee',
    'LivingArea_missing', 'BathroomsTotalInteger_missing',
    'YearBuilt_missing', 'LotSizeAcres_missing', 'DaysOnMarket_anomaly'
]

target_col = 'ClosePrice'

model_df = df[['ListingKey', 'CloseDate_parsed', target_col] + feature_cols].copy()
model_df = model_df.dropna(subset=feature_cols)  # safety net; should be ~0 rows dropped here
print(f'Final model-ready rows: {len(model_df):,}')

Final model-ready rows: 411,419


## 8. Normalization
I'm scaling numeric features with StandardScaler, but I'm **not** fitting the
scaler yet — that has to happen inside the train/test split in Week 4, fit on
the training window only, to avoid leaking test-period info into the scaling.
Fitting it here on the full dataset would be a subtle leak. So this notebook
saves the *unscaled* cleaned CSV, and Week 4 handles the scaler per split.

## 9. Train/Test Split Function (window length is tunable — see Week 4)
Setting this up here since it's a Week 3 deliverable, but the actual experiment
over window lengths (and the model fit/eval) happens in Week 4.

In [42]:
def get_train_test_split(frame, test_month, window_months):
    """
    test_month: pd.Period, e.g. pd.Period('2026-06', freq='M')
    window_months: int, number of months immediately preceding test_month to use as training
    """
    frame = frame.copy()
    frame['YearMonth'] = frame['CloseDate_parsed'].dt.to_period('M')

    test_df = frame[frame['YearMonth'] == test_month]

    train_start = test_month - window_months
    train_df = frame[(frame['YearMonth'] >= train_start) & (frame['YearMonth'] < test_month)]

    return train_df.drop(columns='YearMonth'), test_df.drop(columns='YearMonth')

# quick sanity check with a 12-month window
test_month = pd.Period('2026-06', freq='M')
train_check, test_check = get_train_test_split(model_df, test_month, 12)
print(f'Train rows: {len(train_check):,} | Test rows: {len(test_check):,}')

Train rows: 130,060 | Test rows: 12,841


In [43]:
model_df.to_csv(data_folder + '\\cleaned_full.csv', index=False)
print('Saved cleaned_full.csv')

Saved cleaned_full.csv


## Summary
- Dropped rows with missing/non-positive ClosePrice (unrecoverable target) and
  rows missing Latitude/Longitude (small %, no reasonable impute).
- Imputed LivingArea, BathroomsTotalInteger, YearBuilt (median) and LotSizeAcres
  (City median, then global median) with `_missing` flag columns — no more
  blanket-dropping across 9 columns like last time.
- Clipped negative DaysOnMarket at 0 and flagged as anomaly rather than trusting
  the raw value.
- Capped ClosePrice outliers using the Week 2 IQR bounds instead of just noting
  them.
- Engineered PropertyAge from YearBuilt.
- Excluded ListPrice/OriginalListPrice from features to avoid circularity in a
  Zestimate-style model — flag if you want this decision reversed.
- Built `get_train_test_split()` with a tunable window length; actual window
  experiment happens in Week 4.
- Saved `cleaned_full.csv` for Week 4.